In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys, os, time, copy, csv, evaluate, torch

from glob import glob
from datasets import Dataset, Value, concatenate_datasets
from sklearn.metrics import f1_score, mean_squared_error

from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          BertForSequenceClassification, BertModel,
                          PreTrainedModel, Trainer, TrainingArguments)

#from value_disagreement.datasets import (RedditAnnotatedDataset, ValueEvalDataset,
#                              ValueNetDataset)
#from value_disagreement.datasets.utils import cast_dataset_to_hf, hf_dataset_tokenize
#from value_disagreement.extraction import ValueConstants, ValueTokenizer

sys.path.append('/Proyecto/Value-disagreement/Python/Utilities')
import text_cleansing, Dict_Object


os.environ["TOKENIZERS_PARALLELISM"] = "false"

### VALUENET

In [ ]:
valuenet = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_all.csv",
                         sep='|')
valuenet

In [ ]:
valuenet_pivot = valuenet.pivot(index='scenario', columns='value', values='label').fillna(0).astype(int).reset_index()
valuenet_pivot

In [ ]:
valuenet_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_train.csv",sep=',')
valuenet_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_val.csv",sep=',')
valuenet_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueNET/v0.3_balanced/valuenet_test.csv",sep=',')
valuenet_test_set

### VALUEARG

In [ ]:
valuearg = pd.read_csv(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_all.csv",
                       sep='|')
valuearg.rename(columns={"Argument ID": "uid","variable":"value","value":"label","Premise":"scenario"}, inplace=True)
valuearg.drop_duplicates(subset=['value','scenario'],inplace=True, keep='first', ignore_index=True)
valuearg

In [ ]:
valuearg_pivot = valuearg.pivot(index='scenario', columns='value', values='label').fillna(0).astype(int).reset_index()
valuearg_pivot

In [ ]:
valuenet_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_train.csv",sep=',')
valuenet_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_val.csv",sep=',')
valuenet_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/ValueARG/valuearg_test.csv",sep=',')
valuenet_val_set

### All (valuenet+valuearg)

In [ ]:
value_all = pd.concat([valuenet,valuearg],axis=0, sort = False) ## 2733 rows
#all_dic.drop_duplicates(subset='word',inplace=True, keep='first', ignore_index=True)
value_all

In [ ]:
value_all_train_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_train.csv",sep=',')
value_all_val_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_val.csv",sep=',')
value_all_test_set = np.fromfile(r"/Proyecto/Value-disagreement/Datos/valueALL/value_all_test.csv",sep=',')
value_all_val_set

### BaseLine (valuenet approach)

In [ ]:
def get_splits(df, train_size=0.8, val_size=0.1, test_size=0.1):
    assert train_size + val_size + test_size == 1
    
    num_train = int(len(df)*train_size)
    num_val = int(len(df)*val_size)
    num_test = len(df) - (num_train + num_val)

    idx = np.arange(len(df))
    np.random.shuffle(idx)
    train_set = idx[:num_train]
    val_set = idx[num_train:num_train + num_val]
    test_set = idx[num_train + num_val:]
    
    return train_set, val_set, test_set

In [ ]:
def cast_dataset_to_hf(dataset, split_name, abs_label=True):
    """
    Convert custom dataset to huggingface dataset.
    """

    if abs_label:
        labels = dataset['label'] = [abs(x['label']) for _, x in dataset.iterrows()]
    else:
        labels = dataset['label'] = [x['label'] for _, x in dataset.iterrows()]
        
    dataset_dict = {
        'id': [x['uid'] for _, x in dataset.iterrows()],
        'text': [x['scenario'] for _, x in dataset.iterrows()],
        'orig_label': labels,
        'value': [x['value'] for _, x in dataset.iterrows()],
    }

    hf_dataset = Dataset.from_dict(dataset_dict, split=split_name)
    return hf_dataset

In [ ]:
def hf_dataset_tokenize(hf_dataset, tokenizer, soft_target_type='int'):
    """
    Apply a tokenizer to a huggingface dataset.
    """
    tokenized_dataset = hf_dataset.map(tokenizer.tokenize, batched=True)

    column_names = ['input_ids', 'attention_mask', 'labels']

    tokenized_dataset.set_format(type='torch', columns=column_names)
    
    if soft_target_type == 'float':
        target_type = Value(dtype='float32', id=None)
    else:
        target_type = Value(dtype='int64', id=None)
    
    tokenized_dataset = tokenized_dataset.cast_column('labels', target_type)
    return tokenized_dataset

In [ ]:
class ValueTokenizer():
    def __init__(self, model_name, input_concat=False, label_type="copy", label2id=None):
        self.model_name: str = model_name
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.input_concat = input_concat
        self.label_type = label_type
        self.label2id = None

    def nhot_labels(self, batch_labels, num_classes=10):
        """
        batch_labels contains list of indices, convert to n-hot encoded matrix.
        """
        batch_size = len(batch_labels)
        labels = np.zeros((batch_size, num_classes), dtype=np.float64)
        for i, ex in enumerate(batch_labels):
            for v in ex:
                labels[i, v] = float(1.0)
        return labels

    def tokenize(self, examples):
        # Gather input text
        if self.input_concat:
            batch_size = len(examples['text'])
            batched_inputs = [f"<{examples['value'][i]}> " + f"{self.tokenizer.sep_token} " + examples['text'][i] for i in range(batch_size)]
        else:
            batched_inputs = examples['text']

        # Tokenize~!
        #samples = self.tokenizer(batched_inputs, truncation=True, padding=True)
        samples = self.tokenizer(batched_inputs, truncation=True, padding='max_length', max_length=128)

        # Gather target labels
        if self.label_type == "cast_float":
            samples["labels"] = [float(x) for x in examples["orig_label"]]
        elif self.label_type == "cast_nhot":
            samples['labels'] = self.nhot_labels(examples['labels'], num_classes=10)
        elif self.label_type == "cast_nhot_schwartz":
            batch_labels = []
            for multi_labels in examples['schwartz_labels']:
                batch_labels.append([self.label2id[x] for x in multi_labels])
            samples['labels'] = self.nhot_labels(batch_labels, num_classes=len(self.label2id))
        elif self.label_type == "copy":
            samples['labels'] = examples['labels']
        return samples

In [ ]:
def model_init(checkpoint, tokenizer):
    """
    Initialize the model with the checkpoint and tokenizer.
    """
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=1)
    model.resize_token_embeddings(len(tokenizer.tokenizer))
    return model

In [ ]:
MODEL_NAME = "bert-base-uncased"

In [ ]:
train_hf = cast_dataset_to_hf(valuenet.filter(valuenet_train_set.astype(np.int64), axis=0), "train", abs_label=False)
val_hf = cast_dataset_to_hf(valuenet.filter(valuenet_val_set.astype(np.int64), axis=0), "val", abs_label=False)
test_hf = cast_dataset_to_hf(valuenet.filter(valuenet_test_set.astype(np.int64), axis=0), "test", abs_label=False)
val_hf[3]

In [ ]:
value_tokenizer = ValueTokenizer(
    MODEL_NAME,
    input_concat=True,
    label_type="cast_float"
)

In [ ]:
# Tokenize data
tokenized_dataset_train = hf_dataset_tokenize(train_hf, value_tokenizer, soft_target_type='float')
tokenized_dataset_val = hf_dataset_tokenize(val_hf, value_tokenizer, soft_target_type='float')
tokenized_dataset_test = hf_dataset_tokenize(test_hf, value_tokenizer, soft_target_type='float')
tokenized_dataset_val[3]

In [ ]:
# Add special value tokens and initalize model
num_added_tokens = value_tokenizer.tokenizer.add_special_tokens(
    {"additional_special_tokens": [f"<{x}>" for x in Dict_Object.ValueConstants.SCHWARTZ_VALUES]})
print(f"Added {num_added_tokens} special value tokens")

In [ ]:
#os.environ["WANDB_DISABLED"] = "true"

In [ ]:
model = model_init(MODEL_NAME, value_tokenizer)
#model = model_init("checkpoint-495/", value_tokenizer)

In [ ]:
batch_size = 8
#metric = evaluate.load("mean_squared_error")#"mse")

"""training_args = TrainingArguments(
    output_dir="qiu_trainer",
    learning_rate=5e-06,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    #report_to="wandb",
    report_to="none",
    metric_for_best_model="mse",
    logging_strategy='steps',
    logging_steps=20,
    load_best_model_at_end=True,
    weight_decay=0.00,
)"""

# Memory efficient ChatGPT 
training_args = TrainingArguments(
    output_dir="qiu_trainer",
    learning_rate=2e-5,                # slightly higher for faster convergence with smaller batches
    per_device_train_batch_size=batch_size,     # ↓ batch size greatly reduces memory
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,                # fewer epochs = less training time/memory
    eval_strategy="epoch",             # evaluate after each epoch
    save_strategy="epoch",             # save after each epoch
    logging_strategy="steps",
    logging_steps=50,                  # log less frequently to reduce overhead
    metric_for_best_model="mse",       # or any custom metric you use
    load_best_model_at_end=True,
    weight_decay=0.01,                 # apply light regularization
    fp16=True,                         # use mixed precision (only if using GPU with CUDA)
    gradient_accumulation_steps=2,     # simulate larger batch size without full memory use
    save_total_limit=1,                # only keep best checkpoint to save disk space
    report_to="none"                   # turn off external logging like W&B
)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    rmse = mean_squared_error(y_true=labels, y_pred=predictions, squared=False)
    #rmse = metric.compute(predictions=predictions, references=labels, squared=False)
    return {"rmse": rmse}

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_dataset_train,
    eval_dataset=tokenized_dataset_val,
    #tokenizer=value_tokenizer.tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
def evaluate_model(checkpoint, dataset):
    model = model_init(checkpoint, value_tokenizer)
    args = TrainingArguments(
        output_dir=".",
        do_train=False,
        do_eval=False,
        do_predict=True,
        per_device_eval_batch_size=2
    )

    trainer = Trainer(
        model,
        args,
        tokenizer=value_tokenizer.tokenizer,
    )

    predictions = trainer.predict(dataset)
    pred_int = np.round(predictions.predictions, 0)

    f1_simple_rounding = f1_score(y_true=np.abs(predictions.label_ids), y_pred=np.abs(pred_int), average='macro')
    print(f"F1 score simple rounding: {f1_simple_rounding:.3f}")

In [ ]:
evaluate_model("checkpoint-495/", tokenized_dataset_test)

In [ ]:
dataset_eval = ValueEvalDataset(
    f"{proj_dir}/data/valueeval/dataset-identifying-the-human-values-behind-arguments/",
    cast_to_valuenet=True,
    return_predefined_splits=True
)

test_set_idx = []

for i, elem in enumerate(dataset_eval):
    if elem['split'] == 'test':
        test_set_idx.append(i)

test_set = dataset_eval[test_set_idx]
test_hf = cast_dataset_to_hf(test_set, "test", abs_label=False)

value_tokenizer = ValueTokenizer(
    MODEL_NAME,
    input_concat=True,
    label_type="cast_float"
)

tokenized_dataset_test_eval = hf_dataset_tokenize(test_hf, value_tokenizer, MODEL_NAME, soft_target_type='float')

# Add special value tokens and initalize model
num_added_tokens = value_tokenizer.tokenizer.add_special_tokens(
    {"additional_special_tokens": [f"<{x}>" for x in ValueConstants.SCHWARTZ_VALUES]})
print(f"Added {num_added_tokens} special value tokens")

evaluate_model("checkpoint-495/", tokenized_dataset_test_eval)

In [ ]:
big_ds = concatenate_datasets([tokenized_dataset_test, tokenized_dataset_test_eval])
evaluate_model("checkpoint-495/", big_ds)

In [ ]:
lengths = [len(value_tokenizer.tokenizer.tokenize(x)) for x in valuenet["scenario"]]
pd.Series(lengths).describe()